In [2]:
import gurobipy as gp
from gurobipy import GRB
import os
import pandas as pd
directorio=r"C:\Users\melco\OneDrive\Escritorio\pasantia_imple\datos_TEST"
d={}
B_num={}
p={1:869.434,2:1533.228,3:2430.901,4:3169.766,5:3947.629,6:4690.218,7:5514.84,8:6051.184,9:6693.546,
   10:7264.281,11:8095.383,12:8530.091,13:9088.648,14:9600.03,15:10486.354,16:11616.937,17:12249.156,
   18:12605.908,19:14312.13,20:15063.13,21:16036.058,22:16509.946,23:17643.719,24:18142.476,25:19010.222}
idx_p=gp.tuplelist(p)
maxi=0


for j, archivo in enumerate(os.listdir(directorio)):
    if archivo.endswith('.csv'):
        cont=0
        ruta_archivo = os.path.join(directorio, archivo)
        df = pd.read_csv(ruta_archivo)
        df['Distance'] = df['Distance'].fillna(0)
        df['dist_Acum'] = df['Distance'].cumsum()
        filtered_df = df[df['Speed[m/s]']==0]
        aux=df['dist_Acum'].iloc[-1]
        if aux>=maxi:
            maxi=aux
        k=0
        for i, row in filtered_df.iterrows():
            k=k+1
            d[(j+1, k)] = row['dist_Acum']
            cont=cont+1
            B_num[j+1]=cont
        #print(j)

B = {}

M=maxi
# Iterar sobre el diccionario original
for clave, valor in d.items():
    primer_valor = clave[0]
    if primer_valor not in B:
        B[primer_valor] = {}
    B[primer_valor][clave] = valor
    
B_max=max(list(B_num.values()))
print(B_max)

print(M)
idx_d=gp.tuplelist(d)
#print(d)
#print(p)
#print(B[1])
print(B_num)
idx_B=gp.tuplelist(B_num)
print(idx_B)
#print(idx_d)
S=sum(B_num.values())
print(S)
print(B_num)

11
38036.11949999995
{1: 3, 2: 11}
[1, 2]
14
{1: 3, 2: 11}


In [ ]:
m = gp.Model()
x=m.addVars(idx_d,idx_p, vtype = GRB.CONTINUOUS, name="x")
y=m.addVars(idx_B, vtype=GRB.CONTINUOUS, name="y")
z=m.addVars(idx_d,vtype=GRB.CONTINUOUS, name="z")
m.setObjective(z.sum("*","*"), GRB.MINIMIZE)#-x.sum("*","*","*")
S=sum(B_num.values())
m.addConstrs((x.sum(i,"*","*")>=25*0.9 for i in idx_B),"max_base")
m.addConstrs((x.sum(i,j,"*")<=1 for (i,j) in idx_d ),"max_punto") #for i in idx_B for j in range(1,B_max+1)
m.addConstrs((z[i,j]>=d[i,j]+y[i]-p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_inf")
m.addConstrs((z[i,j]>=-d[i,j]-y[i]+p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_sup")
#m.addConstr(x.sum("*","*","*")>=(0.1*S),"num_datos")
m.addConstrs((x.sum(i,"*",k)<=1 for i in idx_B for k in idx_p),"unicidad")
m.optimize()

In [ ]:
m.write("test.lp")
if m.SolCount > 0:
    # Recuperar los valores de las variables
    vx = m.getAttr('x', x)
    vy = m.getAttr('x', y) 
    vz = m.getAttr('x', z)
    for k in idx_p:
        #print('\nFlujos optimos para {}:'.format(k))
        for i,j in idx_d:
            if vx[i,j,k] > 0:
                print("Base de datos {} punto elegido {} ({}) para la parada {}.".format(i, j,vx[i,j,k],k))
    for i in idx_B:
        if vy[i]!=0:
            print(f"Desplazamiento {vy[i]}") 
    for i,j in idx_d:
        if vz[i,j]!=0:
            print(f"El error del punto {j} con valor {d[i,j]}en la base {i} es {vz[i,j]}")

In [4]:
m1 = gp.Model()
x=m1.addVars(idx_d,idx_p, vtype = GRB.CONTINUOUS, name="x")
y=m1.addVars(idx_B, vtype=GRB.CONTINUOUS, name="y")
z=m1.addVars(idx_d,vtype=GRB.CONTINUOUS, name="z")
m1.setObjective(z.sum("*","*"), GRB.MINIMIZE)#-x.sum("*","*","*")
S=sum(B_num.values())
m1.addConstrs((x.sum(i,"*","*")>=25*0.1 for i in idx_B),"max_base")
m1.addConstrs((x.sum(i,j,"*")<=1 for (i,j) in idx_d ),"max_punto") #for i in idx_B for j in range(1,B_max+1)
m1.addConstrs((z[i,j]>=d[i,j]+y[i]-p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_inf")
m1.addConstrs((z[i,j]>=-d[i,j]-y[i]+p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_sup")
#m.addConstr(x.sum("*","*","*")>=(0.1*S),"num_datos")
m1.addConstrs((x.sum(i,"*",k)<=1 for i in idx_B for k in idx_p),"unicidad")
m1.addConstrs((x[i,j1,k2] + x[i,j2,k1] <= 1 
     for i in idx_B 
     for j1 in range(1, B_num[i]) 
     for j2 in range(j1+1, B_num[i])  # Asegura que j2 > j1
     for k1 in idx_p 
     for k2 in idx_p if k1 < k2),
    name="puntos_cons")
m1.optimize()
m1.write("modelo_real.lp")

Set parameter Username
Academic license - for non-commercial use only - expires 2025-05-14
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 14566 rows, 366 columns and 30750 nonzeros
Model fingerprint: 0x51667126
Coefficient statistics:
  Matrix range     [1e+00, 4e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 7e+04]
Presolve removed 202 rows and 0 columns
Presolve time: 0.07s
Presolved: 366 rows, 14364 columns, 30144 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.00s

Barrier performed 0 iterations in 0.16 seconds (0.03 work units)
Barrier solve interrupted - model solved by another algorithm


Solved with primal simplex
Iteration    Objective  

In [ ]:
m1.write("test.lp")
if m1.SolCount > 0:
    # Recuperar los valores de las variables
    vx = m1.getAttr('x', x)
    vy = m1.getAttr('x', y) 
    vz = m1.getAttr('x', z)
    for k in idx_p:
        #print('\nFlujos optimos para {}:'.format(k))
        for i,j in idx_d:
            if vx[i,j,k] > 0:
                print("Base de datos {} punto elegido {} ({}) para la parada {}.".format(i, j,vx[i,j,k],k))
    for i in idx_B:
        if vy[i]!=0:
            print(f"Desplazamiento {vy[i]}") 
    for i,j in idx_d:
        if vz[i,j]!=0:
            print(f"El error del punto {j} con valor {d[i,j]}en la base {i} es {vz[i,j]}")